In [21]:
import kagglehub

In [53]:
import numpy as np
import pandas as pd
import os
import cv2
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [23]:
Base_Path = "/kaggle/input/datasets/ranimbenamara/dataset-livrable-1"

Meta_Path = "/kaggle/input/datasets/sohampatra2/data-preparation-and-annotation/Output"

print(Base_Path)
print(Meta_Path)
print("Dataset Exists:", os.path.exists(Base_Path))
print("Metadata Exists:", os.path.exists(Meta_Path))

/kaggle/input/datasets/ranimbenamara/dataset-livrable-1
/kaggle/input/datasets/sohampatra2/data-preparation-and-annotation/Output
Dataset Exists: True
Metadata Exists: True


In [24]:
for folder in os.listdir(Base_Path):
    print(folder)

Dataset Livrable 1 - Schematics
Dataset Livrable 1 - Painting
Dataset Livrable 1 - Sketch
Dataset Livrable 1 - Photo
Dataset Livrable 1 - Text


In [25]:
records = []
for category in os.listdir(Base_Path):

    category_path = os.path.join(Base_Path, category)

    if not os.path.isdir(category_path):
        continue

    label = category.replace("Dataset Livrable 1 - ", "")

    for filename in os.listdir(category_path):

        file_path = os.path.join(category_path, filename)

        if filename.lower().endswith((".jpg",".jpeg",".png")):
            records.append({
                "file": filename,
                "path": file_path,
                "label": label
            })

master_df = pd.DataFrame(records)

print("tots images:", len(master_df))
master_df.head()

tots images: 16705


,file,path,label
0,schematics_05345.jpg,/kaggle/input/datasets/ranimbenamara/dataset-l...,Schematics
1,schematics_03355.jpg,/kaggle/input/datasets/ranimbenamara/dataset-l...,Schematics
2,schematics_03800.jpg,/kaggle/input/datasets/ranimbenamara/dataset-l...,Schematics
3,schematics_01078.jpg,/kaggle/input/datasets/ranimbenamara/dataset-l...,Schematics
4,schematics_01715.jpg,/kaggle/input/datasets/ranimbenamara/dataset-l...,Schematics


In [26]:
master_df["label"].value_counts()

label
Photo         5338
Schematics    4902
Text          4874
Sketch        1379
Painting       212
Name: count, dtype: int64

In [46]:
blurry_df = pd.read_csv(os.path.join(Meta_Path,"blurry_images.csv"))

lowres_df = pd.read_csv(os.path.join(Meta_Path,"low_resolution_images.csv"))

duplicate_df = pd.read_csv(os.path.join(Meta_Path,"near_duplicate_images.csv"))

clip_df = pd.read_csv(os.path.join(Meta_Path,"clip_corrected_labels.csv"))

In [49]:
master_df = master_df[~master_df["file"].isin(blurry_df["Filename"])]

master_df = master_df[~master_df["file"].isin(lowres_df["Filename"])]

master_df = master_df[~master_df["file"].isin(duplicate_df["Image2"])]

print("no. of images after cleaning:", len(master_df))

no. of images after cleaning: 14657


In [50]:
print(master_df.columns)
print(f"(Row, Col) : {master_df.shape}")

Index(['file', 'path', 'label'], dtype='object')
(Row, Col) : (14657, 3)


In [51]:
class inputdata(Dataset):
    def __init__(self,df):
        self.df=df
        self.mean=torch.tensor([0.485,0.456, 0.406]).view(3,1,1)
        self.std=torch.tensor([0.229, 0.224,0.225]).view(3,1,1)

        self.label_map={"Painting": 0,"Photo": 1,"Schematics": 2,"Sketch": 3,"Text": 4}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        img_path = self.df.iloc[i ]["path"]
        labelStr = self.df.iloc[i]["label"]

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img,(128,128))
        img = img.astype(np.float32)/255.0 

        img = torch.from_numpy(img).permute(2,0,1)
        img = (img -self.mean)/self.std

        label = torch.tensor(self.label_map[labelStr], dtype=torch.long)

        return img,label

In [52]:
dataset = inputdata(master_df)
img, label = dataset[6000]

print(img.shape)
print(label)       

torch.Size([3, 128, 128])
tensor(1)


In [64]:
train_df, test_df = train_test_split(master_df,test_size=0.2,
                                     stratify=master_df["label"],random_state=42)

In [65]:
print(f"train :{len(train_df)} , test :{len(test_df)}")

train :11725 , test :2932


In [66]:
train_dataset = inputdata(train_df)
test_dataset  = inputdata(test_df)

In [68]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [69]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 128, 128])
torch.Size([32])


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3, padding=1),
            nn.ReLU(),nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3, padding=1),
            nn.ReLU(),nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3, padding=1),
            nn.ReLU(),nn.MaxPool2d(2,2),

        )
        
        self.classfier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*16*16, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128,5)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classfier(x)
        return x


model = CNN()
print(model)
        

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN().to(device)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for imgs, labels in train_loader:
        outputs = model(imgs)
        loss =criterion(outputs,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")
